In [11]:
import ast
import math
import os
import pickle
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import matthews_corrcoef
from tqdm import tqdm

In [12]:
test_accessions = open("../../../data/processed_data/genome_partitions/test_partition_accessions.txt").read().splitlines()
#test_accessions = ["GCF_001277215.2"] #REMOVE

In [13]:
def process_fragmented_cds(model_preds_dict, model_dict_short_fragments):
    """
    Process fragmented CDS data to convert group labels, add error positions,
    and separate by CDS length. Updates existing model_dict_short_fragments.
    """
    
    for read_name, data in model_preds_dict.items():
        cds_coords = data['cds_coords']
        cds_fragments_connection = data['cds_fragments_connection']
        
        # Step 1: Convert group labels to index-based connections
        new_connections = []
        group_dict = {}
        
        # First pass: collect all group members
        for i, connection in enumerate(cds_fragments_connection):
            if isinstance(connection, list) and len(connection) == 1:
                item = connection[0]
                if isinstance(item, str) and item.startswith('group_'):
                    if item not in group_dict:
                        group_dict[item] = []
                    group_dict[item].append(i)
        
        # Second pass: build new connections
        processed_indices = set()
        for i, connection in enumerate(cds_fragments_connection):
            if i in processed_indices:
                continue
                
            if isinstance(connection, list) and len(connection) == 1:
                item = connection[0]
                if isinstance(item, str) and item.startswith('group_'):
                    # Add the group as a single connection
                    if item in group_dict:
                        new_connections.append(group_dict[item])
                        processed_indices.update(group_dict[item])
                else:
                    # Regular single CDS
                    new_connections.append([item])
                    processed_indices.add(i)
        
        # Step 2: Calculate CDS lengths and process errors
        valid_connections = []
        valid_cds_coords = []
        short_connections = []
        short_cds_coords = []
        seq_errors = []
        short_seq_errors = []
        
        for connection in new_connections:
            # Calculate total CDS length for this connection
            if len(connection) == 1:
                # Single CDS
                cds = cds_coords[connection[0]]
                total_length = cds[1] - cds[0] + 1
                current_errors = []
            else:
                # Connected fragments - calculate total length
                fragments = [(i, cds_coords[i]) for i in connection]
                fragments.sort(key=lambda x: x[1][0])  # Sort by start position
                
                total_length = 0
                current_errors = []
                
                for j, (_, cds) in enumerate(fragments):
                    total_length += cds[1] - cds[0] + 1
                    
                    # Calculate errors between fragments
                    if j < len(fragments) - 1:
                        current_cds = cds
                        next_cds = fragments[j + 1][1]
                        
                        current_frame = int(current_cds[2])
                        next_frame = int(next_cds[2])
                        
                        # Determine indel type by frame shift pattern
                        frame_shift = (next_frame - current_frame) % 3
                        
                        if frame_shift == 1:
                            indel_type = 'I'
                        elif frame_shift == 2:
                            indel_type = 'D'
                        else:
                            continue
                        
                        # Calculate gap midpoint
                        gap_start = current_cds[1] + 1
                        gap_end = next_cds[0] - 1
                        midpoint = (gap_start + gap_end) // 2
                        
                        current_errors.append(f"{midpoint}{indel_type}")
            
            # Decide where to place this CDS based on length
            if total_length > 60:
                # Keep in main dictionary
                valid_connections.append(connection)
                seq_errors.extend(current_errors)
                
            elif total_length >= 30:
                # Move to short fragments dictionary
                short_connections.append(connection)
                short_seq_errors.extend(current_errors)
            
            # CDSs < 30 bp are discarded (not added to either dictionary)
        
        # Update main dictionary
        if valid_connections:
            # Rebuild cds_coords list with only valid CDSs and update indices
            new_cds_coords = []
            index_mapping = {}
            
            for connection in valid_connections:
                for old_idx in connection:
                    if old_idx not in index_mapping:
                        index_mapping[old_idx] = len(new_cds_coords)
                        new_cds_coords.append(cds_coords[old_idx])
            
            # Update connections with new indices
            final_connections = []
            for connection in valid_connections:
                final_connections.append([index_mapping[i] for i in connection])
            
            data['cds_coords'] = new_cds_coords
            data['cds_fragments_connection'] = final_connections
            data['seq_error_positions'] = seq_errors
        else:
            # No valid CDSs, mark for removal from main dict
            model_preds_dict[read_name] = None
        
        # Add to existing short fragments dictionary if applicable
        if short_connections:
            # Initialize if read doesn't exist in short fragments dict
            if read_name not in model_dict_short_fragments:
                model_dict_short_fragments[read_name] = {
                    'cds_coords': [],
                    'cds_fragments_connection': [],
                    'seq_error_positions': []
                }
            
            # Rebuild for short fragments
            new_short_coords = []
            short_index_mapping = {}
            
            # Start indexing from existing cds_coords length
            existing_short_coords = model_dict_short_fragments[read_name]['cds_coords']
            start_idx = len(existing_short_coords)
            
            for connection in short_connections:
                for old_idx in connection:
                    if old_idx not in short_index_mapping:
                        short_index_mapping[old_idx] = start_idx + len(new_short_coords)
                        new_short_coords.append(cds_coords[old_idx])
            
            final_short_connections = []
            for connection in short_connections:
                final_short_connections.append([short_index_mapping[i] for i in connection])
            
            # Append to existing data
            model_dict_short_fragments[read_name]['cds_coords'].extend(new_short_coords)
            model_dict_short_fragments[read_name]['cds_fragments_connection'].extend(final_short_connections)
            model_dict_short_fragments[read_name]['seq_error_positions'].extend(short_seq_errors)
    
    # Remove None entries from main dictionary
    model_preds_dict = {k: v for k, v in model_preds_dict.items() if v is not None and k is not None}
    
    return model_preds_dict, model_dict_short_fragments

In [14]:
def process_model_preds(test_accession, model_type, testset_type, model_preds_path, seq_len):
    """ 
    Process model predictions from a GFF file for a given test accession.

    Args:
        test_accession (str): The accession identifier for the test dataset.
        model_preds_path (str): The path to the model predictions directory.

    Returns:
        model_dict (dict): A dictionary where keys are read names and values are dictionaries with 'cds_coords' (CDS coordinates).
        model_dict_short_fragments (dict): A dictionary for predicted short fragments (<= 60 bps).
    """

    #Initialize
    model_dict = dict()
    model_dict_short_fragments = dict()

    #Read model predictions GFF file
    with open(f"../../../data/processed_data/predictions/raw_predictions/{model_type}/{testset_type}/{model_preds_path}/predictions_{test_accession}.gff", "r") as file:
        file.readline() #Skip first line

        #Get CDS predictions for read
        for line in file:
            read_name = line.split("\t")[0]
            attr_desc = line.split("\t")[8]
            attr_type = line.split("\t")[2]

            if attr_type == "CDS":


                if read_name not in model_dict.keys():
                    model_dict[read_name] = dict()
                    model_dict[read_name]["cds_coords"] = []
                    model_dict[read_name]["cds_fragments_connection"] = []
                    model_dict[read_name]["seq_error_positions"] = [] #ONLY STORE ERRORS WITHIN CDS REGIONS!

                    model_dict_short_fragments[read_name] = dict()
                    model_dict_short_fragments[read_name]["cds_coords"] = []
                    model_dict_short_fragments[read_name]["cds_fragments_connection"] = []
                    model_dict_short_fragments[read_name]["seq_error_positions"] = [] #ONLY STORE ERRORS WITHIN CDS REGIONS!

                    counter_cds_on_read = 0
                    counter_cds_on_read_short = 0

                #Get CDS coordinates and reading frame
                cds_start = int(line.split("\t")[3])
                cds_end = int(line.split("\t")[4])
                if cds_end == seq_len:
                    cds_end -= 3

                rf = str(int(line.split("\t")[7]))

                cds_coords = [cds_start, cds_end, rf]

                #If "group_id" is present, an indel has been predicted
                if "group_id" in attr_desc:
                    group_id = attr_desc.split("group_id=")[1].split(".")[0]
                    model_dict[read_name]["cds_coords"].append(cds_coords)
                    model_dict[read_name]["cds_fragments_connection"].append([group_id])
                
                else: 
                    if cds_end - cds_start + 1 > 60: 
                        model_dict[read_name]["cds_coords"].append(cds_coords)
                        model_dict[read_name]["cds_fragments_connection"].append([counter_cds_on_read])
                        counter_cds_on_read += 1

                    #store predicted CDS sequences longer than 30 bps in short fragments dict
                    else:
                        if cds_end - cds_start + 1 > 30: 
                            model_dict_short_fragments[read_name]["cds_coords"].append(cds_coords)
                            model_dict_short_fragments[read_name]["cds_fragments_connection"].append([counter_cds_on_read_short])
                            counter_cds_on_read_short += 1


    #####MOVE SHORT FRAGMENTS TO DICT AND INTRODUCE SEQUENCING ERRORS INFORMATION
    model_dict, model_dict_short_fragments = process_fragmented_cds(model_dict, model_dict_short_fragments)

    #Reorganize to only include reads with coding sequences
    model_dict = {
        read_name: data 
        for read_name, data in model_dict.items() 
        if data["cds_coords"]  #Check if cds_coords is not empty
    }

    model_dict_short_fragments = {
        read_name: data 
        for read_name, data in model_dict_short_fragments.items() 
        if data["cds_coords"]  #Check if cds_coords is not empty
    }


    return model_dict, model_dict_short_fragments

# Process preds for data without sequencing errors

In [ ]:
#Navigate to DeepCDS without sequencing errors predictions directory
model_type = "DeepCDS/model_without_errors"

#Process preds for model trained on 100, 200, and 400 genomes (only 300bp dataset)
testset_types = ["without_errors_300bp"]
model_preds_paths = ["full_model_100_genomes_seed_42_trained_final", "full_model_200_genomes_seed_42_trained_final", "full_model_400_genomes_seed_42_trained_final"]

for testset_type in testset_types:
    print(testset_type)

    seq_len = int(testset_type.split("_")[-1].strip("bp"))

    for model_preds_path in tqdm(model_preds_paths, desc="Processing predictions for model type..."):
        for test_accession in test_accessions:
            os.makedirs(f"../../../data/processed_data/predictions/processed_predictions/{model_type}/{testset_type}/{model_preds_path}/{test_accession}", exist_ok=True)
            
            model_preds_dict, model_preds_dict_short_fragments = process_model_preds(test_accession, model_type, testset_type, model_preds_path, seq_len)

            with open(f"../../../data/processed_data/predictions/processed_predictions/{model_type}/{testset_type}/{model_preds_path}/{test_accession}/model_preds_dict.pkl", "wb") as processed_preds_file:
                pickle.dump(model_preds_dict, processed_preds_file)

            with open(f"../../../data/processed_data/predictions/processed_predictions/{model_type}/{testset_type}/{model_preds_path}/{test_accession}/model_preds_dict_short_fragments.pkl", "wb") as processed_preds_file:
                pickle.dump(model_preds_dict_short_fragments, processed_preds_file)


#Process preds for model trained on all genomes 
testset_types = ["without_errors_60bp",
                "without_errors_75bp",
                "without_errors_100bp",
                "without_errors_150bp",
                "without_errors_300bp",
                "without_errors_700bp",
                "without_errors_1000bp"]


model_preds_paths = ["full_model_all_genomes_seed_42_trained_final"]

#TODO: PROCESS PREDS WHEN MODEL HAS FINISHED TRAINING!

without_errors_300bp


Processing predictions for model type...: 100%|██████████| 3/3 [01:15<00:00, 25.14s/it]


# Process preds for data with sequencing errors

In [ ]:
model_types = ["DeepCDS/model_with_errors", "DeepCDS/model_with_substitution_errors"]

testset_types = [WRITE HERE]

model_preds_paths = ["full_model_100_genomes_seed_42_trained_final"]

for testset_type in testset_types:
    print(testset_type)
    
    seq_len = int(testset_type.split("_")[-1].strip("bp"))

    for model_type in model_types:
        
        for model_preds_path in model_preds_paths:
            for test_accession in test_accessions:
                os.makedirs(f"../../../data/processed_data/predictions/processed_predictions/{model_type}/{testset_type}/{model_preds_path}/{test_accession}", exist_ok=True)
                
                model_preds_dict, model_preds_dict_short_fragments = process_model_preds(test_accession, model_type, testset_type, model_preds_path, seq_len)

                with open(f"../../../data/processed_data/predictions/processed_predictions/{model_type}/{testset_type}/{model_preds_path}/{test_accession}/model_preds_dict.pkl", "wb") as processed_preds_file:
                    pickle.dump(model_preds_dict, processed_preds_file)

                with open(f"../../../data/processed_data/predictions/processed_predictions/{model_type}/{testset_type}/{model_preds_path}/{test_accession}/model_preds_dict_short_fragments.pkl", "wb") as processed_preds_file:
                    pickle.dump(model_preds_dict_short_fragments, processed_preds_file)

without_errors_60bp


NameError: name 'seq_len' is not defined